# NB13 — Nivel 2: EDA exploratorio y arquitectura del modelo personalizado RUNA/Strava

**Objetivo**: explorar los patrones de FC (frecuencia cardíaca) y ritmo en nuestros atletas reales
(datos de Strava vía Supabase) y diseñar la arquitectura del Nivel 2 del modelo jerárquico.

**Contexto metodológico (tesis)**:
- Nivel 1: prior poblacional entrenado con Endomondo (NB12) → artefacto `nivel1_prior_poblacional.pkl`
- Nivel 2: personalización por atleta usando sus propias actividades — HR + pace como señales primarias
- **Consentimiento**: todos los atletas firmaron el formulario de onboarding con consentimiento
  explícito para uso académico anonimizado (datos sin nombre ni cédula en el modelo).
- **N total activos**: ~27 atletas en el portal, ~21 con ≥8 semanas y ≥10 runs con HR disponible

**Estructura del notebook**:
- **Parte A — EDA exploratorio**: carga desde Supabase, perfiles de atletas, distribuciones HR/pace,
  correlaciones, boxplots por sexo/edad, heatmap semanas, primer Ridge test LOAO-CV
- **Parte B — Arquitectura Nivel 2**: definición formal de features, pipeline de entrenamiento,
  protocolo LOAO-CV (Leave-One-Athlete-Out), conformal intervals, integración con Nivel 1

## 0 · Setup y carga de entorno

In [ ]:
import os, sys, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# --- Paths ---
BASE = Path(r'C:/Users/andre/OneDrive/Documentos/Maestría Analítica Aplicada/running_coaching')
OUT_DIR = BASE / 'ml' / 'notebooks' / 'outputs' / 'nb13'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Agregar src/ al path para usar supabase_client
sys.path.insert(0, str(BASE))
load_dotenv(BASE / '.env')

# --- Estilo RUNA ---
CRIMSON = '#C41E3A'
NAVY    = '#1F4B99'
CREAM   = '#FDFBF7'
SLATE   = '#4B5563'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'sans-serif',
    'axes.prop_cycle': plt.cycler(color=[CRIMSON, NAVY, '#2563EB', '#059669', '#D97706', '#7C3AED']),
})

print('Setup OK — BASE:', BASE.exists())

## A · 1 — Carga de datos desde Supabase

In [ ]:
from src.storage.supabase_client import get_client

client = get_client()
assert client is not None, "Supabase no configurado — verifica .env"

# ── 1a. Perfiles de atletas ─────────────────────────────────────────────────
profiles_raw = (
    client.table("athlete_profiles")
    .select("cedula,age,sex,experience_years,weekly_km_current,main_sport,weekly_days")
    .execute()
).data or []

df_profiles = pd.DataFrame(profiles_raw)
print(f"athlete_profiles: {len(df_profiles)} atletas")
print(df_profiles.dtypes)
df_profiles.head()

In [ ]:
# ── 1b. Actividades (solo runs con HR) ─────────────────────────────────────
# La columna raw es JSONB — Supabase lo retorna ya como dict
acts_raw = (
    client.table("activities")
    .select("strava_id,cedula,sport_type,activity_date,distance_m,duration_sec,"
            "elevation_m,avg_pace_sec_km,raw")
    .eq("sport_type", "Run")
    .order("activity_date", desc=False)
    .execute()
).data or []

print(f"Actividades Run cargadas: {len(acts_raw)}")

rows = []
for r in acts_raw:
    raw = r.get("raw") or {}
    rows.append({
        "strava_id":       r["strava_id"],
        "cedula":          r["cedula"],
        "activity_date":   r["activity_date"],
        "distance_m":      r.get("distance_m"),
        "distance_km":     (r["distance_m"] / 1000.0) if r.get("distance_m") else None,
        "duration_sec":    r.get("duration_sec"),
        "duration_min":    (r["duration_sec"] / 60.0) if r.get("duration_sec") else None,
        "elevation_m":     r.get("elevation_m"),
        "pace_sec_km":     r.get("avg_pace_sec_km"),
        "pace_min_km":     (r["avg_pace_sec_km"] / 60.0) if r.get("avg_pace_sec_km") else None,
        "avg_hr":          raw.get("average_heartrate"),
        "max_hr":          raw.get("max_heartrate"),
        "avg_cadence":     raw.get("average_cadence"),
        "avg_speed_ms":    raw.get("average_speed"),
        "suffer_score":    raw.get("suffer_score"),
        "perceived_exertion": raw.get("perceived_exertion"),
    })

df_acts = pd.DataFrame(rows)
df_acts["activity_date"] = pd.to_datetime(df_acts["activity_date"], utc=True).dt.tz_localize(None)

print(f"\nShape: {df_acts.shape}")
print(f"Atletas únicos: {df_acts['cedula'].nunique()}")
print(f"\nCobertura HR (avg_hr no nulo): {df_acts['avg_hr'].notna().sum()} "
      f"({df_acts['avg_hr'].notna().mean()*100:.1f}%)")
df_acts.describe(percentiles=[.1,.25,.5,.75,.9]).round(1)

In [ ]:
# ── 1c. Weekly features desde Supabase ─────────────────────────────────────
wf_raw = (
    client.table("weekly_features")
    .select("cedula,week_start,total_km,total_runs,avg_pace_sec_km,avg_hr,"
            "ctl,atl,tsb,acwr,readiness_score")
    .order("week_start", desc=False)
    .execute()
).data or []

df_wf = pd.DataFrame(wf_raw)
if not df_wf.empty:
    df_wf["week_start"] = pd.to_datetime(df_wf["week_start"])
    numeric_cols = ["total_km","total_runs","avg_pace_sec_km","avg_hr","ctl","atl","tsb","acwr","readiness_score"]
    for c in numeric_cols:
        if c in df_wf.columns:
            df_wf[c] = pd.to_numeric(df_wf[c], errors="coerce")

print(f"weekly_features: {len(df_wf)} filas, {df_wf['cedula'].nunique() if not df_wf.empty else 0} atletas")
if not df_wf.empty:
    print(f"Rango temporal: {df_wf['week_start'].min().date()} → {df_wf['week_start'].max().date()}")
    print(df_wf.describe().round(2))

In [ ]:
# ── 1d. Merge actividades + perfiles → dataset maestro ─────────────────────
df = df_acts.merge(df_profiles, on="cedula", how="left")

# Filtros de coherencia
df = df[
    df["distance_km"].between(1.0, 60.0) &
    df["pace_min_km"].between(3.0, 12.0) &
    df["duration_min"].between(5.0, 300.0)
].copy()

# HR max estimado por edad (Tanaka: 208 - 0.7*age)
df["hr_max_est"] = np.where(df["age"].notna(), 208 - 0.7 * df["age"], np.nan)
df["pct_hr_max"] = np.where(
    df["avg_hr"].notna() & df["hr_max_est"].notna(),
    df["avg_hr"] / df["hr_max_est"],
    np.nan
)

# Zona HR (según % FCmax Karvonen clásico)
def zona_hr(pct):
    if pd.isna(pct): return np.nan
    if pct < 0.60: return 1
    if pct < 0.70: return 2
    if pct < 0.80: return 3
    if pct < 0.90: return 4
    return 5

df["zona_hr"] = df["pct_hr_max"].apply(zona_hr)

# Anonymize: usar índice numérico en lugar de cédula para figuras
cedula_to_idx = {c: i+1 for i, c in enumerate(sorted(df["cedula"].unique()))}
df["athlete_id"] = df["cedula"].map(cedula_to_idx)

print(f"Dataset maestro: {len(df)} actividades, {df['cedula'].nunique()} atletas")
print(f"\nActividades con HR: {df['avg_hr'].notna().sum()} ({df['avg_hr'].notna().mean()*100:.1f}%)")
print(f"Actividades con zona HR: {df['zona_hr'].notna().sum()}")
df[["pace_min_km","avg_hr","pct_hr_max","zona_hr","distance_km","age","sex"]].describe().round(2)

## A · 2 — Perfil demográfico del grupo

In [ ]:
# Resumen por atleta: cuántas actividades, HR coverage, rango temporal
per_athlete = (
    df.groupby("cedula").agg(
        n_runs=("strava_id", "count"),
        n_hr=("avg_hr", "count"),
        first_run=("activity_date", "min"),
        last_run=("activity_date", "max"),
        age=("age", "first"),
        sex=("sex", "first"),
        avg_pace_min_km=("pace_min_km", "mean"),
        avg_hr_mean=("avg_hr", "mean"),
        median_distance_km=("distance_km", "median"),
    ).reset_index()
)
per_athlete["weeks_span"] = (per_athlete["last_run"] - per_athlete["first_run"]).dt.days / 7
per_athlete["pct_hr"] = per_athlete["n_hr"] / per_athlete["n_runs"] * 100
per_athlete["athlete_id"] = per_athlete["cedula"].map(cedula_to_idx)

# ── Elegibles para Nivel 2 ──────────────────────────────────────────────────
WEEKS_MIN = 8
HR_RUNS_MIN = 10

eligible = per_athlete[
    (per_athlete["weeks_span"] >= WEEKS_MIN) &
    (per_athlete["n_hr"] >= HR_RUNS_MIN)
].copy()

print(f"Atletas totales: {len(per_athlete)}")
print(f"Elegibles Nivel 2 (≥{WEEKS_MIN} sem + ≥{HR_RUNS_MIN} runs con HR): {len(eligible)}")
print()
display_cols = ["athlete_id","sex","age","n_runs","n_hr","pct_hr","weeks_span","avg_pace_min_km","avg_hr_mean"]
per_athlete[display_cols].sort_values("n_runs", ascending=False).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Perfil demográfico — Atletas RUNA', fontsize=13, fontweight='bold')

# Distribución de edad
ax = axes[0]
ax.hist(per_athlete["age"].dropna(), bins=10, color=CRIMSON, alpha=0.8, edgecolor='white')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('N atletas')
ax.set_title('Distribución de edad')
ax.axvline(per_athlete["age"].median(), color=NAVY, lw=2, ls='--',
           label=f'Mediana {per_athlete["age"].median():.0f}a')
ax.legend(fontsize=9)

# Distribución por sexo
ax = axes[1]
sex_counts = per_athlete["sex"].value_counts()
ax.bar(sex_counts.index, sex_counts.values, color=[CRIMSON, NAVY], alpha=0.85)
for i, (label, val) in enumerate(sex_counts.items()):
    ax.text(i, val + 0.2, str(val), ha='center', fontweight='bold')
ax.set_title('Distribución por sexo')
ax.set_ylabel('N atletas')

# Actividades y cobertura HR por atleta
ax = axes[2]
pa_sorted = per_athlete.sort_values("n_runs", ascending=True)
ax.barh(pa_sorted["athlete_id"].astype(str), pa_sorted["n_runs"],
        alpha=0.35, color=NAVY, label="Runs totales")
ax.barh(pa_sorted["athlete_id"].astype(str), pa_sorted["n_hr"],
        alpha=0.85, color=CRIMSON, label="Runs con HR")
ax.set_xlabel("N actividades")
ax.set_title("Actividades por atleta")
ax.legend(fontsize=9)
ax.set_ylabel("ID atleta (anónimo)")

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig1_demografia.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 3 — Distribuciones de ritmo y FC

In [ ]:
df_hr = df[df["avg_hr"].notna() & df["pace_min_km"].notna()].copy()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribuciones de ritmo y FC — Actividades RUNA', fontsize=13, fontweight='bold')

# 1. Histograma ritmo global
ax = axes[0, 0]
ax.hist(df["pace_min_km"].dropna(), bins=40, color=CRIMSON, alpha=0.8, edgecolor='white')
ax.set_xlabel('Ritmo (min/km)')
ax.set_title('Distribución global de ritmo')
ax.axvline(df["pace_min_km"].median(), color=NAVY, lw=2, ls='--',
           label=f'Mediana: {df["pace_min_km"].median():.1f} min/km')
ax.legend(fontsize=9)

# 2. Histograma FC media
ax = axes[0, 1]
ax.hist(df_hr["avg_hr"], bins=40, color=NAVY, alpha=0.8, edgecolor='white')
ax.set_xlabel('FC media (bpm)')
ax.set_title('Distribución global de FC media')
ax.axvline(df_hr["avg_hr"].median(), color=CRIMSON, lw=2, ls='--',
           label=f'Mediana: {df_hr["avg_hr"].median():.0f} bpm')
ax.legend(fontsize=9)

# 3. % FCmax
ax = axes[0, 2]
pct_vals = df_hr["pct_hr_max"].dropna() * 100
ax.hist(pct_vals, bins=40, color='#059669', alpha=0.8, edgecolor='white')
ax.set_xlabel('% FCmax estimada (Tanaka)')
ax.set_title('Intensidad relativa de sesiones')
for z, label, color in [(60,'Z1','#93C5FD'), (70,'Z2','#6EE7B7'), (80,'Z3','#FCD34D'),
                         (90,'Z4','#F87171'), (100,'Z5','#C41E3A')]:
    ax.axvline(z, color=color, lw=1.5, alpha=0.7)
ax.text(62, ax.get_ylim()[1]*0.9, 'Z1-Z5', fontsize=8, color=SLATE)

# 4. Boxplot ritmo por sexo
ax = axes[1, 0]
df_sex = df[df["sex"].notna()]
sexes = df_sex["sex"].unique()
data_by_sex = [df_sex[df_sex["sex"] == s]["pace_min_km"].dropna().values for s in sexes]
bp = ax.boxplot(data_by_sex, patch_artist=True,
                boxprops=dict(facecolor=CRIMSON, alpha=0.6),
                medianprops=dict(color=NAVY, lw=2))
ax.set_xticks(range(1, len(sexes)+1))
ax.set_xticklabels(sexes)
ax.set_ylabel('Ritmo (min/km)')
ax.set_title('Ritmo por sexo')

# 5. Boxplot FC por sexo
ax = axes[1, 1]
data_hr_by_sex = [df_sex[df_sex["sex"] == s]["avg_hr"].dropna().values for s in sexes]
ax.boxplot(data_hr_by_sex, patch_artist=True,
           boxprops=dict(facecolor=NAVY, alpha=0.6),
           medianprops=dict(color=CRIMSON, lw=2))
ax.set_xticks(range(1, len(sexes)+1))
ax.set_xticklabels(sexes)
ax.set_ylabel('FC media (bpm)')
ax.set_title('FC media por sexo')

# 6. Distribución zonas HR
ax = axes[1, 2]
zona_counts = df_hr["zona_hr"].dropna().value_counts().sort_index()
zone_colors = ['#93C5FD', '#6EE7B7', '#FCD34D', '#F87171', '#C41E3A']
bars = ax.bar([f'Z{int(z)}' for z in zona_counts.index], zona_counts.values,
              color=zone_colors[:len(zona_counts)], alpha=0.85)
ax.set_xlabel('Zona HR (% FCmax Karvonen)')
ax.set_ylabel('N actividades')
ax.set_title('Distribución de zonas de intensidad')
for bar, val in zip(bars, zona_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val}\n({val/len(df_hr)*100:.0f}%)', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig2_distribuciones.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 4 — Relación FC–Ritmo: señal central del Nivel 2

In [ ]:
from scipy.stats import pearsonr, spearmanr

df_hr_clean = df_hr.dropna(subset=["avg_hr", "pace_min_km", "pct_hr_max"])

r_pearson, p_pearson = pearsonr(df_hr_clean["avg_hr"], df_hr_clean["pace_min_km"])
r_spearman, p_spearman = spearmanr(df_hr_clean["avg_hr"], df_hr_clean["pace_min_km"])
r_pct, p_pct = pearsonr(df_hr_clean["pct_hr_max"], df_hr_clean["pace_min_km"])

print(f"Correlación FC media ↔ ritmo:")
print(f"  Pearson r = {r_pearson:.3f}  (p={p_pearson:.2e})")
print(f"  Spearman ρ = {r_spearman:.3f}  (p={p_spearman:.2e})")
print(f"\nCorrelación %FCmax ↔ ritmo (ritmo relativo):")
print(f"  Pearson r = {r_pct:.3f}  (p={p_pct:.2e})")
print(f"\nInterpretación:")
print(f"  {'Relación positiva significativa' if r_pearson > 0.1 and p_pearson < 0.05 else 'Relación débil'} entre FC media y ritmo")
print(f"  (A ritmo más lento = sesiones de mayor FC? Mixto: afectan distancia, fatiga, terreno)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Relación FC–Ritmo en atletas RUNA (actividad por actividad)', fontsize=12, fontweight='bold')

# 1. Scatter global FC vs ritmo (coloreado por atleta)
ax = axes[0]
n_athletes = df_hr_clean["athlete_id"].nunique()
colors_scatter = plt.cm.tab20(np.linspace(0, 1, n_athletes))
for i, (aid, grp) in enumerate(df_hr_clean.groupby("athlete_id")):
    ax.scatter(grp["avg_hr"], grp["pace_min_km"], alpha=0.4, s=15,
               color=colors_scatter[i % len(colors_scatter)], label=f'A{aid}' if n_athletes <= 15 else None)
# Línea de tendencia global
from numpy.polynomial.polynomial import polyfit as polyfit_np
coefs = np.polyfit(df_hr_clean["avg_hr"], df_hr_clean["pace_min_km"], 1)
x_line = np.linspace(df_hr_clean["avg_hr"].min(), df_hr_clean["avg_hr"].max(), 100)
ax.plot(x_line, np.polyval(coefs, x_line), color='black', lw=2, ls='--',
        label=f'Tendencia (r={r_pearson:.2f})')
ax.set_xlabel('FC media (bpm)')
ax.set_ylabel('Ritmo (min/km)')
ax.set_title(f'FC media vs Ritmo\n(N={len(df_hr_clean)} actividades)')
ax.legend(fontsize=7, ncol=2 if n_athletes > 10 else 1)

# 2. Scatter %FCmax vs ritmo
ax = axes[1]
ax.scatter(df_hr_clean["pct_hr_max"] * 100, df_hr_clean["pace_min_km"],
           alpha=0.4, s=15, color=NAVY)
coefs2 = np.polyfit(df_hr_clean["pct_hr_max"], df_hr_clean["pace_min_km"], 1)
x2 = np.linspace(df_hr_clean["pct_hr_max"].min(), df_hr_clean["pct_hr_max"].max(), 100)
ax.plot(x2 * 100, np.polyval(coefs2, x2), color=CRIMSON, lw=2, ls='--',
        label=f'Tendencia (r={r_pct:.2f})')
ax.set_xlabel('% FCmax estimada')
ax.set_ylabel('Ritmo (min/km)')
ax.set_title(f'%FCmax vs Ritmo\n(N={len(df_hr_clean)})')
ax.legend(fontsize=9)

# 3. FC media por zona vs ritmo medio de la zona
ax = axes[2]
zona_summary = df_hr_clean.groupby("zona_hr").agg(
    n=("avg_hr","count"), avg_hr=("avg_hr","mean"), avg_pace=("pace_min_km","mean")
).reset_index()
ax.scatter(zona_summary["avg_hr"], zona_summary["avg_pace"],
           s=zona_summary["n"]*2, color=CRIMSON, alpha=0.8, zorder=5)
for _, row in zona_summary.iterrows():
    ax.annotate(f'Z{int(row["zona_hr"])}\n(n={int(row["n"])})',
                (row["avg_hr"], row["avg_pace"]),
                textcoords='offset points', xytext=(5, 5), fontsize=9)
ax.set_xlabel('FC media de la zona (bpm)')
ax.set_ylabel('Ritmo medio (min/km)')
ax.set_title('Resumen por zona de intensidad\n(tamaño = N actividades)')

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig3_hr_vs_ritmo.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 5 — Variabilidad inter-atleta: por qué necesitamos el Nivel 2

In [ ]:
# Correlación HR-ritmo POR ATLETA — muestra heterogeneidad inter-atleta
corrs_by_athlete = []
for ced, grp in df_hr_clean.groupby("cedula"):
    if len(grp) >= 5:
        r, p = pearsonr(grp["avg_hr"], grp["pace_min_km"])
        corrs_by_athlete.append({
            "athlete_id": cedula_to_idx[ced],
            "n": len(grp),
            "r_hr_pace": r,
            "p": p,
            "avg_pace": grp["pace_min_km"].mean(),
            "avg_hr": grp["avg_hr"].mean(),
        })

df_corrs = pd.DataFrame(corrs_by_athlete).sort_values("r_hr_pace")

print("Correlación FC↔Ritmo por atleta (>= 5 actividades con HR):")
print(df_corrs[["athlete_id","n","r_hr_pace","p","avg_pace","avg_hr"]].to_string(index=False))
print(f"\nRango: r = [{df_corrs['r_hr_pace'].min():.2f}, {df_corrs['r_hr_pace'].max():.2f}]")
print(f"Mediana: r = {df_corrs['r_hr_pace'].median():.2f}")
print(f"\n→ Alta variabilidad inter-atleta justifica personalización (Nivel 2)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Variabilidad inter-atleta — por qué personalizar', fontsize=12, fontweight='bold')

# Izquierda: barras de correlación por atleta
ax = axes[0]
colors_bar = [CRIMSON if r < 0 else NAVY for r in df_corrs["r_hr_pace"]]
bars = ax.barh([f'A{aid}' for aid in df_corrs["athlete_id"]],
               df_corrs["r_hr_pace"], color=colors_bar, alpha=0.8)
ax.axvline(0, color='black', lw=1)
ax.axvline(df_corrs["r_hr_pace"].median(), color='orange', lw=2, ls='--',
           label=f'Mediana r={df_corrs["r_hr_pace"].median():.2f}')
ax.set_xlabel('Pearson r (FC ↔ Ritmo)')
ax.set_title('Correlación FC-Ritmo por atleta\n(mismo feature, respuesta diferente)')
ax.legend(fontsize=9)

# Derecha: curvas FC-ritmo por atleta (regresión lineal por atleta)
ax = axes[1]
x_global = np.linspace(df_hr_clean["avg_hr"].min(), df_hr_clean["avg_hr"].max(), 100)
for i, (ced, grp) in enumerate(df_hr_clean.groupby("cedula")):
    if len(grp) >= 5:
        coef = np.polyfit(grp["avg_hr"], grp["pace_min_km"], 1)
        y_pred = np.polyval(coef, x_global)
        mask = (x_global >= grp["avg_hr"].min()) & (x_global <= grp["avg_hr"].max())
        ax.plot(x_global[mask], y_pred[mask], alpha=0.7, lw=1.5,
                color=colors_scatter[i % len(colors_scatter)],
                label=f'A{cedula_to_idx[ced]}')

ax.set_xlabel('FC media (bpm)')
ax.set_ylabel('Ritmo predicho (min/km)')
ax.set_title('Pendientes FC→Ritmo por atleta\n(cada línea = un atleta)')
if df_hr_clean["athlete_id"].nunique() <= 15:
    ax.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig4_variabilidad_interatleta.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 6 — Heatmap semanal: evolución de la carga por atleta

In [ ]:
if df_wf.empty:
    print("Sin weekly_features en Supabase — saltando heatmap")
else:
    df_wf["athlete_id"] = df_wf["cedula"].map(cedula_to_idx)
    df_wf["week_label"] = df_wf["week_start"].dt.strftime("%Y-%m-%d")

    # Pivot: atletas en eje Y, semanas en eje X, valor = CTL o total_km
    pivot_ctl = df_wf.pivot_table(index="athlete_id", columns="week_label", values="ctl", aggfunc="mean")
    pivot_km = df_wf.pivot_table(index="athlete_id", columns="week_label", values="total_km", aggfunc="sum")

    fig, axes = plt.subplots(2, 1, figsize=(16, 8))
    fig.suptitle('Evolución semanal por atleta (RUNA)', fontsize=13, fontweight='bold')

    # CTL heatmap
    ax = axes[0]
    # Mostrar solo las últimas 24 semanas para que sea legible
    cols = pivot_ctl.columns[-24:]
    sns.heatmap(pivot_ctl[cols], ax=ax, cmap='YlOrRd', cbar_kws={'label': 'CTL'},
                linewidths=0.5, linecolor='white', annot=False,
                yticklabels=[f'A{i}' for i in pivot_ctl.index])
    ax.set_title('CTL (Carga Crónica) por atleta y semana')
    ax.set_xlabel('Semana')
    ax.set_ylabel('Atleta')
    # Rotar labels del eje X
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

    # km heatmap
    ax = axes[1]
    sns.heatmap(pivot_km[cols], ax=ax, cmap='Blues', cbar_kws={'label': 'km/semana'},
                linewidths=0.5, linecolor='white', annot=False,
                yticklabels=[f'A{i}' for i in pivot_km.index])
    ax.set_title('Volumen semanal (km) por atleta')
    ax.set_xlabel('Semana')
    ax.set_ylabel('Atleta')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=7)

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'nb13_fig5_heatmap_semanal.png', dpi=150, bbox_inches='tight')
    plt.show()

## A · 7 — Primer modelo Ridge: baseline Nivel 2 con LOAO-CV

**LOAO-CV** (Leave-One-Athlete-Out Cross-Validation): para cada atleta en el conjunto
de test, entrenamos con todos los demás. Esto simula exactamente la situación de
predecir el ritmo de un atleta nuevo dado su FC + perfil.

Esto es el **baseline del Nivel 2**: si Ridge ya supera la mediana global, hay señal.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_absolute_error, r2_score

# ── Dataset para modelado ────────────────────────────────────────────────────
# Solo atletas con HR, perfil de edad/sexo
df_model = df_hr_clean.dropna(subset=["avg_hr","pct_hr_max","zona_hr","pace_min_km","age","sex"]).copy()
df_model["sex_bin"] = (df_model["sex"].str.upper() == "M").astype(float)
df_model["zona_hr"] = df_model["zona_hr"].astype(float)

FEATURES_V1 = ["avg_hr", "pct_hr_max", "zona_hr", "distance_km",
               "duration_min", "elevation_m", "age", "sex_bin"]
# Rellenar elevation NaN con 0
for col in FEATURES_V1:
    if col in df_model.columns:
        df_model[col] = pd.to_numeric(df_model[col], errors="coerce").fillna(0)

TARGET = "pace_min_km"
GROUP = "cedula"

# Filtrar atletas con al menos 5 actividades en el modelo
counts = df_model.groupby("cedula").size()
eligible_cedulas = counts[counts >= 5].index
df_model = df_model[df_model["cedula"].isin(eligible_cedulas)].copy()

X = df_model[FEATURES_V1].values
y = df_model[TARGET].values
groups = df_model[GROUP].values

print(f"Dataset LOAO-CV:")
print(f"  Actividades: {len(df_model)}")
print(f"  Atletas: {len(np.unique(groups))}")
print(f"  Features: {FEATURES_V1}")
print(f"  Target: {TARGET} (ritmo min/km)")
print(f"  Rango target: [{y.min():.2f}, {y.max():.2f}]")

In [ ]:
logo = LeaveOneGroupOut()
preds_ridge = np.zeros_like(y, dtype=float)
preds_naive = np.zeros_like(y, dtype=float)
mae_by_athlete = {}

for tr_idx, te_idx in logo.split(X, y, groups):
    scaler = StandardScaler().fit(X[tr_idx])
    model = Ridge(alpha=1.0)
    model.fit(scaler.transform(X[tr_idx]), y[tr_idx])
    preds_ridge[te_idx] = model.predict(scaler.transform(X[te_idx]))
    preds_naive[te_idx] = np.median(y[tr_idx])  # baseline: mediana del training

    # MAE por atleta (para el plot por atleta)
    ced = np.unique(groups[te_idx])[0]
    mae_by_athlete[cedula_to_idx[ced]] = {
        "mae_ridge": mean_absolute_error(y[te_idx], preds_ridge[te_idx]) * 60,  # a sec/km
        "mae_naive": mean_absolute_error(y[te_idx], preds_naive[te_idx]) * 60,
        "n": len(te_idx),
    }

# Métricas globales
mae_ridge_s = mean_absolute_error(y, preds_ridge) * 60
mae_naive_s = mean_absolute_error(y, preds_naive) * 60
r2_ridge = r2_score(y, preds_ridge)
r2_naive = r2_score(y, preds_naive)

print("=" * 50)
print("RESULTADOS LOAO-CV — Nivel 2 baseline")
print("=" * 50)
print(f"  Baseline (mediana):   MAE = {mae_naive_s:.1f} sec/km   R² = {r2_naive:.3f}")
print(f"  Ridge (V1):           MAE = {mae_ridge_s:.1f} sec/km   R² = {r2_ridge:.3f}")
print(f"  Mejora vs baseline:   {mae_naive_s - mae_ridge_s:+.1f} sec/km ({(1 - mae_ridge_s/mae_naive_s)*100:.1f}%)")
print()
# MAE por atleta
df_mae = pd.DataFrame(mae_by_athlete).T.reset_index().rename(columns={"index": "athlete_id"})
df_mae = df_mae.sort_values("mae_ridge")
print("MAE por atleta (sec/km):")
print(df_mae.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('LOAO-CV — Ridge Nivel 2 vs Baseline', fontsize=12, fontweight='bold')

# 1. Predicho vs Real
ax = axes[0]
ax.scatter(y, preds_ridge, alpha=0.5, s=20, color=NAVY, label='Ridge V1')
ax.scatter(y, preds_naive, alpha=0.3, s=10, color=SLATE, label='Baseline (mediana)')
min_v, max_v = min(y.min(), preds_ridge.min()), max(y.max(), preds_ridge.max())
ax.plot([min_v, max_v], [min_v, max_v], 'k--', lw=1.5, label='Perfecto')
ax.set_xlabel('Ritmo real (min/km)')
ax.set_ylabel('Ritmo predicho (min/km)')
ax.set_title(f'Predicho vs Real\nRidge MAE={mae_ridge_s:.0f} s/km · Baseline={mae_naive_s:.0f} s/km')
ax.legend(fontsize=9)

# 2. MAE por atleta: Ridge vs baseline
ax = axes[1]
x_pos = np.arange(len(df_mae))
w = 0.35
ax.bar(x_pos - w/2, df_mae["mae_naive"], w, color=SLATE, alpha=0.7, label='Baseline')
ax.bar(x_pos + w/2, df_mae["mae_ridge"], w, color=CRIMSON, alpha=0.8, label='Ridge V1')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'A{int(a)}' for a in df_mae["athlete_id"]], rotation=45, ha='right')
ax.set_ylabel('MAE (sec/km)')
ax.set_title('MAE por atleta (LOAO-CV)')
ax.legend(fontsize=9)
ax.axhline(mae_ridge_s, color=CRIMSON, ls='--', lw=1.5, alpha=0.7, label=f'Media Ridge={mae_ridge_s:.0f}')

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig6_loao_cv_ridge.png', dpi=150, bbox_inches='tight')
plt.show()

## A · 8 — Importancia de features (coeficientes Ridge)

In [ ]:
# Entrenar Ridge en dataset completo para ver coeficientes (interpretabilidad)
scaler_full = StandardScaler().fit(X)
ridge_full = Ridge(alpha=1.0).fit(scaler_full.transform(X), y)

coef_df = pd.DataFrame({
    "feature": FEATURES_V1,
    "coef": ridge_full.coef_,
    "abs_coef": np.abs(ridge_full.coef_),
}).sort_values("abs_coef", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors_coef = [CRIMSON if c < 0 else NAVY for c in coef_df["coef"]]
ax.barh(coef_df["feature"], coef_df["coef"], color=colors_coef, alpha=0.85)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("Coeficiente Ridge (espacio estandarizado)")
ax.set_title("Importancia de features — Ridge Nivel 2\n(entrenado en dataset completo)")

# Anotaciones
for _, row in coef_df.iterrows():
    ax.text(row["coef"] + (0.002 if row["coef"] >= 0 else -0.002),
            row.name, f'{row["coef"]:.3f}', va='center',
            ha='left' if row["coef"] >= 0 else 'right', fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'nb13_fig7_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nIntercept: {ridge_full.intercept_:.3f}")
print(f"\nInterpretación:")
for _, row in coef_df.sort_values("abs_coef", ascending=False).iterrows():
    direction = "↑ pace (más lento)" if row["coef"] > 0 else "↓ pace (más rápido)"
    print(f"  {row['feature']:20s}: coef={row['coef']:+.3f} → {direction}")

---

# PARTE B — Arquitectura del Nivel 2

## B · 1 — Definición formal del modelo jerárquico

El modelo de RUNA sigue una arquitectura de **dos niveles**:

### Nivel 1 (NB12): Prior poblacional
- Fuente: Endomondo / FitRec — ~800 usuarios, ~40K sesiones
- Features: género, FCmax observada, FC media, % FCmax, zona HR
- Target: pace (min/km)
- Modelo: Ridge / GBM (mejor en LOUO-CV)
- Artefacto: `nivel1_prior_poblacional.pkl`

### Nivel 2 (este notebook): Personalización
- Fuente: nuestros atletas RUNA/Strava (N~21 con HR)
- Features: Nivel 1 + CTL/ATL/TSB + historial semanal + demografía local
- Target: pace individual
- Protocolo evaluación: **LOAO-CV** (Leave-One-Athlete-Out)
- Integración: fine-tuning del prior OR stack (Nivel 1 como feature extra)

### Relación con la tesis
```
Pregunta de investigación:
  "¿En qué medida la incorporación de datos fisiológicos longitudinales
   individuales (FC + carga de entrenamiento) mejora la estimación del
   ritmo de carrera respecto a modelos poblacionales?"

H₀: MAE_Nivel2 ≥ MAE_Nivel1_LOAO  (sin ganancia de personalización)
H₁: MAE_Nivel2 < MAE_Nivel1_LOAO  (la personalización reduce error)
```

## B · 2 — Feature engineering para Nivel 2

### Features candidatos (por categoría)

| Categoría | Variable | Fuente | Justificación |
|-----------|----------|--------|---------------|
| **Intensidad sesión** | `avg_hr` | Strava/activities.raw | Intensidad directa de la sesión |
| **Intensidad relativa** | `pct_hr_max` | calculado (Tanaka) | Normaliza por condición del atleta |
| **Zona HR** | `zona_hr` | calculado | Categoría de esfuerzo |
| **Carga aguda** | `atl` | weekly_features | Fatiga reciente (últimas 1-2 semanas) |
| **Carga crónica** | `ctl` | weekly_features | Fitness base (últimas 6 semanas) |
| **Balance** | `tsb` | weekly_features | TSB = CTL - ATL (readiness) |
| **Relación carga** | `acwr` | weekly_features | Proxy riesgo lesión / intensidad relativa |
| **Morfología sesión** | `distance_km` | activities | Distancia del run |
| **Desnivel** | `elevation_m` | activities | Terreno (afecta ritmo) |
| **Demografía** | `age`, `sex_bin` | athlete_profiles | Control demográfico |
| **Experiencia** | `experience_years` | athlete_profiles | Control nivel |
| **Volumen base** | `weekly_km_current` | athlete_profiles | Volumen declarado |

### Features que NO se incluirán en Nivel 2
- **PRs declarados**: sesgo de recuerdo alto, no correlacionan bien con rendimiento real
- **Predicciones Riegel del atleta**: son el target del Nivel 1, no feature del 2
- **Datos Endomondo**: solo para prior; contaminaría la evaluación

### Protocolo LOAO-CV
```
Para cada atleta i en {1..N}:
    train = todos los atletas j ≠ i
    test  = atleta i (todas sus actividades)
    →  El modelo NUNCA ve datos del atleta i durante el entrenamiento
    →  Evalúa capacidad de generalización a atletas nuevos
```

Esta es la métrica relevante para la tesis: **¿puede el modelo predecir
el ritmo de un atleta que no conoce, dado su FC y perfil?**

## B · 3 — Feature set V2: incorporar CTL/ATL/TSB desde weekly_features

In [ ]:
if df_wf.empty:
    print("Sin weekly_features — omitiendo V2. Ejecutar pipeline para todos los atletas primero.")
    df_v2 = df_model.copy()  # fallback a V1
    FEATURES_V2 = FEATURES_V1
else:
    # Asignar semana ISO a cada actividad
    df_model2 = df_model.copy()
    df_model2["week_start"] = df_model2["activity_date"].dt.to_period("W-SUN").dt.start_time

    # Join con weekly_features: tomar la semana más reciente disponible
    wf_subset = df_wf[["cedula","week_start","ctl","atl","tsb","acwr"]].copy()
    df_v2 = df_model2.merge(wf_subset, on=["cedula","week_start"], how="left")

    # Rellenar NaN con 0 (atletas sin weekly_features aún)
    for c in ["ctl","atl","tsb","acwr"]:
        df_v2[c] = pd.to_numeric(df_v2[c], errors="coerce").fillna(0)

    coverage = df_v2[["ctl","atl"]].notna().all(axis=1).mean()
    print(f"Cobertura CTL/ATL/TSB en V2: {coverage*100:.1f}% de actividades")
    print(f"Actividades con wf match: {(df_v2['ctl'] > 0).sum()}/{len(df_v2)}")

    FEATURES_V2 = FEATURES_V1 + ["ctl", "atl", "tsb", "acwr"]
    print(f"\nFeatures V2: {FEATURES_V2}")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

def run_loao_cv(X_in, y_in, groups_in, model_fn, name=""):
    """Corre LOAO-CV y retorna predicciones + MAE global."""
    logo = LeaveOneGroupOut()
    preds = np.zeros_like(y_in, dtype=float)
    for tr, te in logo.split(X_in, y_in, groups_in):
        scaler = StandardScaler().fit(X_in[tr])
        mdl = model_fn()
        mdl.fit(scaler.transform(X_in[tr]), y_in[tr])
        preds[te] = mdl.predict(scaler.transform(X_in[te]))
    mae_s = mean_absolute_error(y_in, preds) * 60
    r2 = r2_score(y_in, preds)
    print(f"  {name:30s}: MAE = {mae_s:.1f} sec/km   R² = {r2:.3f}")
    return preds, mae_s, r2

# Dataset V2
X2 = df_v2[FEATURES_V2].values
y2 = df_v2[TARGET].values
groups2 = df_v2[GROUP].values

# Filtrar mismo criterio
counts2 = pd.Series(groups2).value_counts()
eligible2 = counts2[counts2 >= 5].index
mask2 = pd.Series(groups2).isin(eligible2).values
X2, y2, groups2 = X2[mask2], y2[mask2], groups2[mask2]

print(f"Dataset V2: {len(X2)} actividades, {len(np.unique(groups2))} atletas")
print("=" * 65)
print("COMPARACIÓN LOAO-CV: V1 vs V2 (con CTL/ATL/TSB)")
print("=" * 65)

_, mae_v1, r2_v1 = run_loao_cv(X, y, groups, lambda: Ridge(alpha=1.0), "Ridge V1 (sin carga)")
_, mae_v2, r2_v2 = run_loao_cv(X2, y2, groups2, lambda: Ridge(alpha=1.0), "Ridge V2 (con CTL/ATL/TSB)")
_, mae_gb, r2_gb = run_loao_cv(X2, y2, groups2,
                                lambda: GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=0),
                                "GradientBoosting V2")

print(f"\nMejora V1→V2 (Ridge): {mae_v1 - mae_v2:+.1f} sec/km")
print(f"Mejora V1→GB:         {mae_v1 - mae_gb:+.1f} sec/km")

## B · 4 — Guardar artefacto Nivel 2 y resumen de resultados

In [ ]:
import pickle, datetime

# Reentrenar modelo final con todos los datos
scaler_v2 = StandardScaler().fit(X2)
best_model_v2 = Ridge(alpha=1.0).fit(scaler_v2.transform(X2), y2)

artefacto_v2 = {
    "model": best_model_v2,
    "scaler": scaler_v2,
    "features": FEATURES_V2,
    "target": TARGET,
    "n_athletes": int(len(np.unique(groups2))),
    "n_activities": int(len(X2)),
    "loao_cv_results": {
        "ridge_v1_mae_sec_km": float(mae_v1),
        "ridge_v2_mae_sec_km": float(mae_v2),
        "gradient_boosting_v2_mae_sec_km": float(mae_gb),
        "ridge_v1_r2": float(r2_v1),
        "ridge_v2_r2": float(r2_v2),
        "gradient_boosting_v2_r2": float(r2_gb),
    },
    "dataset_source": "RUNA/Strava — atletas con consentimiento explícito",
    "version": "2.0-exploratory",
    "date": datetime.date.today().isoformat(),
    "note": "Datos anonimizados — sin nombre/cédula en el artefacto"
}

out_pkl = OUT_DIR / 'nivel2_runa_v2_exploratory.pkl'
with open(out_pkl, 'wb') as f:
    pickle.dump(artefacto_v2, f)
print(f"Artefacto Nivel 2 guardado: {out_pkl}")

# Resumen JSON (para tesis)
resumen = {
    "notebook": "NB13",
    "fecha": datetime.date.today().isoformat(),
    "n_atletas_total": len(per_athlete),
    "n_atletas_elegibles_nivel2": len(eligible),
    "n_atletas_con_hr": int((per_athlete["n_hr"] >= HR_RUNS_MIN).sum()),
    "actividades_totales": len(df),
    "actividades_con_hr": int(df["avg_hr"].notna().sum()),
    "correlacion_hr_ritmo_global": {"pearson_r": float(r_pearson), "spearman_rho": float(r_spearman)},
    "loao_cv": {
        "baseline_mae_sec_km": float(mae_naive_s),
        "ridge_v1_mae_sec_km": float(mae_v1),
        "ridge_v2_mae_sec_km": float(mae_v2),
        "gradient_boosting_v2_mae_sec_km": float(mae_gb),
    }
}

with open(OUT_DIR / 'nb13_resultados.json', 'w') as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("RESUMEN NB13 — Nivel 2 RUNA")
print("=" * 60)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

## B · 5 — Próximos pasos y decisiones pendientes

### Resultados clave del EDA (completar tras ejecutar)
- N atletas total / elegibles / con HR — ver tabla A.2
- Correlación global HR↔ritmo: r = [ver celda A.4]
- MAE baseline: X sec/km vs MAE Ridge V1: Y sec/km vs Ridge V2: Z sec/km
- ¿HR + CTL/ATL mejoran respecto a solo HR? → cuantificado en B.3

### Decisiones de arquitectura para versión final (NB13b)
1. **¿Ridge vs GBM?** Ridge si MAE similar (interpretabilidad > potencia); GBM si mejora >10 sec/km
2. **¿Incluir prior Nivel 1?** Stack: `pace_pred_nivel1` como feature adicional en V2
3. **¿Conformal intervals?** Split-conformal sobre atletas del hold-out (α=0.2 → 80% cobertura)
4. **¿Umbral N mínimo?** Atleta necesita ≥5 actividades con HR para que el modelo sea confiable
5. **¿Filtrar privacidad HR?** Algunos atletas tienen HR null por Strava privacy settings

### Limitaciones a documentar en la tesis
- N pequeño (N~21) → alta varianza en LOAO-CV por atleta
- Sesgo de selección: atletas que conectan Strava y usan HR son más comprometidos
- FCmax estimada por Tanaka (208-0.7*age) tiene ±10-15 bpm de error individual
- CTL/ATL calculados desde actividades de Strava (running solo) — no incluye otros deportes
- Privacidad Strava: algunos atletas bloquean HR en su configuración → HR null

### Umbral de publicación para tesis
- **Publicable ahora** (NB13 exploratorio): EDA + correlaciones + Ridge V1 LOAO-CV
- **Publicable cuando N≥40**: comparación estadística Friedman-Nemenyi entre modelos
- **Publicable con NB14**: conformal intervals + integración con Nivel 1 prior